# Embeddings, Block-Embeddings & RAG — eine kleine Demo

Dieses Notebook zeigt in drei Schritten, wie moderne Retrieval-Systeme funktionieren — komplett **lokal**, ohne API-Key:

1. **Embeddings** — Text in Vektoren umwandeln und Ähnlichkeit messen.
2. **Block-Embeddings** — ein längeres Dokument in Blöcke (Chunks) zerlegen und jeden Block einzeln einbetten.
3. **RAG** (Retrieval-Augmented Generation) — zu einer Frage die passenden Blöcke suchen und ein lokales Sprachmodell damit eine fundierte Antwort erzeugen lassen.

Verwendete Bausteine:
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2` (Hugging Face, lokal)
- Generierung: **Ollama** (lokaler Server, z. B. `gemma3:4b`)

## Setup

Abhängigkeiten installieren. `sentence-transformers` liefert das Embedding-Modell, `ollama` ist der Python-Client für den lokalen Ollama-Server.

Voraussetzung: [Ollama](https://ollama.com) ist installiert und läuft (`ollama serve`), und das gewünschte Modell ist gezogen, z. B. `ollama pull gemma3:4b`.

In [ ]:
%pip install -q sentence-transformers ollama numpy

## 1. Embeddings — Text als Vektor

Ein Embedding-Modell bildet jeden Text auf einen Vektor (hier 384 Dimensionen) ab. Semantisch ähnliche Texte liegen im Vektorraum nah beieinander. Wir messen die Nähe mit der **Kosinus-Ähnlichkeit** (1.0 = identisch, 0.0 = unabhängig).

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Kleines, schnelles HF-Modell (~90 MB). Beim ersten Aufruf wird es heruntergeladen.
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

saetze = [
    "Die Katze schläft auf dem Sofa.",
    "Ein Stubentiger ruht sich auf der Couch aus.",
    "Der Aktienmarkt ist heute gefallen.",
]

# Jeder Satz wird zu einem Vektor. normalize_embeddings=True -> Vektoren haben Länge 1,
# dann ist das Skalarprodukt direkt die Kosinus-Ähnlichkeit.
vektoren = model.encode(saetze, normalize_embeddings=True)
print("Form der Embedding-Matrix:", vektoren.shape)  # (3 Sätze, 384 Dimensionen)

In [ ]:
def kosinus(a, b):
    return float(np.dot(a, b))  # Vektoren sind normalisiert -> Skalarprodukt = Kosinus

print("Katze/Sofa  vs.  Stubentiger/Couch :", round(kosinus(vektoren[0], vektoren[1]), 3))
print("Katze/Sofa  vs.  Aktienmarkt       :", round(kosinus(vektoren[0], vektoren[2]), 3))

Obwohl Satz 1 und 2 kaum gemeinsame Wörter haben, ist ihre Ähnlichkeit deutlich höher als zum Satz über den Aktienmarkt — genau das macht Embeddings für die Suche wertvoll.

## 2. Block-Embeddings — ein Dokument in Chunks zerlegen

Ein ganzes Dokument in einen einzigen Vektor zu pressen verliert Details. Stattdessen zerlegt man den Text in **Blöcke** (Chunks) — z. B. Absätze — und bettet jeden Block einzeln ein. So kann man später gezielt die relevanten Stellen finden, statt das ganze Dokument.

Hier ein kleines Wissensdokument, aufgeteilt in Absatz-Blöcke:

In [ ]:
dokument = """
Der Wasserkreislauf beschreibt die kontinuierliche Bewegung des Wassers auf, über und unter der Erdoberfläche. Angetrieben wird er von der Sonnenenergie.

Bei der Verdunstung geht flüssiges Wasser aus Ozeanen, Seen und Böden in den gasförmigen Zustand über und steigt als Wasserdampf in die Atmosphäre auf.

In der Höhe kühlt der Wasserdampf ab und kondensiert zu winzigen Tröpfchen, die zusammen Wolken bilden. Dieser Vorgang heißt Kondensation.

Werden die Tröpfchen zu schwer, fallen sie als Niederschlag zu Boden – etwa als Regen, Schnee oder Hagel.

Ein Teil des Niederschlags versickert im Boden und speist das Grundwasser, ein anderer Teil fließt oberirdisch zurück in Flüsse und Meere.
""".strip()

# Einfaches Chunking: pro Absatz ein Block. In der Praxis nutzt man oft
# überlappende Fenster fester Token-Länge.
bloecke = [b.strip() for b in dokument.split("\n\n") if b.strip()]
print(f"{len(bloecke)} Blöcke:\n")
for i, b in enumerate(bloecke):
    print(f"[{i}] {b[:70]}...")

In [ ]:
# Jeden Block einbetten -> das ist unser Mini-"Vektor-Index".
block_vektoren = model.encode(bloecke, normalize_embeddings=True)
print("Index-Form:", block_vektoren.shape)  # (Anzahl Blöcke, 384)

### Retrieval: passende Blöcke zu einer Frage finden

Die Frage wird mit demselben Modell eingebettet und mit allen Block-Vektoren verglichen. Die Blöcke mit der höchsten Ähnlichkeit sind die relevantesten.

In [ ]:
def suche(frage, top_k=2):
    frage_vec = model.encode([frage], normalize_embeddings=True)[0]
    scores = block_vektoren @ frage_vec  # Ähnlichkeit zu jedem Block
    top = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i]), bloecke[i]) for i in top]

for idx, score, text in suche("Wie entstehen Wolken?"):
    print(f"[Block {idx}  score={score:.3f}] {text}\n")

## 3. RAG — Retrieval + lokales Sprachmodell (Ollama)

**RAG** kombiniert die Suche aus Schritt 2 mit einem Sprachmodell: Wir holen die relevanten Blöcke, geben sie dem Modell als Kontext und lassen es die Frage **nur auf Basis dieses Kontexts** beantworten. Das reduziert Halluzinationen und macht die Antwort nachvollziehbar.

Als Generator dient ein Modell auf dem lokalen **Ollama-Server** (Standard: `http://localhost:11434`). Welches Modell verwendet wird, steuert die Variable `MODELL` — mit `ollama list` siehst du, was bei dir verfügbar ist.

In [ ]:
import ollama

# Modell auf dem lokalen Ollama-Server. Alternativen (falls gezogen):
#   "llama3.2:1b"  -> sehr schnell, schwächeres Deutsch
#   "qwen2.5:14b"  -> beste Qualität, langsamer
MODELL = "gemma3:4b"

# Kurzer Verbindungs-Check: läuft der Server und ist das Modell da?
verfuegbar = [m.model for m in ollama.list().models]
print("Verfügbare Ollama-Modelle:", verfuegbar)
assert MODELL in verfuegbar, f"'{MODELL}' fehlt — bitte `ollama pull {MODELL}` ausführen."

In [ ]:
def rag_antwort(frage, top_k=2):
    # 1. Retrieval: relevante Blöcke holen
    treffer = suche(frage, top_k=top_k)
    kontext = "\n\n".join(f"[Block {idx}] {text}" for idx, _, text in treffer)

    # 2. Generation: das Ollama-Modell mit dem Kontext antworten lassen
    antwort = ollama.chat(
        model=MODELL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Beantworte die Frage AUSSCHLIESSLICH anhand des gelieferten Kontexts. "
                    "Steht die Antwort nicht im Kontext, sage das offen."
                ),
            },
            {"role": "user", "content": f"Kontext:\n{kontext}\n\nFrage: {frage}"},
        ],
        options={"temperature": 0},  # deterministisch(er) für die Demo
    )
    return antwort.message.content, treffer

antwort, treffer = rag_antwort("Warum steigt Wasser in die Atmosphäre auf?")
print("Antwort:\n", antwort)
print("\nGenutzte Blöcke:", [idx for idx, _, _ in treffer])

In [ ]:
# Gegenprobe: eine Frage, deren Antwort NICHT im Dokument steht.
antwort, _ = rag_antwort("Wie heiß ist die Sonnenoberfläche?")
print(antwort)

## Zusammenfassung

| Schritt | Was passiert | Werkzeug |
|---|---|---|
| **Embeddings** | Text → Vektor, Ähnlichkeit per Kosinus | `all-MiniLM-L6-v2` (Hugging Face, lokal) |
| **Block-Embeddings** | Dokument in Chunks zerlegen, jeden einbetten → Index | `all-MiniLM-L6-v2` |
| **RAG** | relevante Blöcke suchen + Modell generiert Antwort aus dem Kontext | Ollama (z. B. `gemma3:4b`) |

**Nächste Schritte für echte Projekte:**
- Überlappende Chunks fester Token-Länge statt nur Absätze.
- Eine Vektordatenbank (FAISS, Chroma, pgvector) statt eines NumPy-Arrays.
- Stärkere Embeddings für Deutsch: z. B. `intfloat/multilingual-e5-small`.
- Größeres Ollama-Modell für bessere Antworten: einfach `MODELL = "qwen2.5:14b"` setzen.
- Quellenangaben in der Antwort, indem die Block-Nummern im Prompt mit zitiert werden.